In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint


import matplotlib.pyplot as plt

In [2]:
qu_df = pd.read_excel("dataset/widsdatathon2025/TRAIN_CORRECTED/TRAIN_QUANTITATIVE_METADATA_new.xlsx")
cat_df = pd.read_excel("dataset/widsdatathon2025/TRAIN_CORRECTED/TRAIN_CATEGORICAL_METADATA_new.xlsx")
df_sol = pd.read_excel("dataset/widsdatathon2025/TRAIN_CORRECTED/TRAINING_SOLUTIONS.xlsx")
data_dict = pd.read_excel("dataset/widsdatathon2025/Data Dictionary.xlsx")


# qu_df = qu_df.drop(index=qu_df[qu_df["MRI_Track_Age_at_Scan"] < 3].index)
# # qu_df = qu_df.drop(index=qu_df.query('MRI_Track_Age_at_Scan >= 15').index)
# qu_df = qu_df.drop(index=qu_df[qu_df["MRI_Track_Age_at_Scan"].isna() == True].index)


# cat_df = cat_df.drop(index=cat_df[cat_df["participant_id"].isin(qu_df["participant_id"].values) == False].index)
# df_sol = df_sol.drop(index=df_sol[df_sol["participant_id"].isin(qu_df["participant_id"])== False].index)

qu_df = pd.merge(df_sol, qu_df, how="outer")
# qu_df.drop(columns=["MRI_Track_Age_at_Scan"])

qu_df = qu_df.dropna()
print(qu_df.dtypes)
# cat_df = pd.merge(df_sol, cat_df, on="participant_id", how="outer")

# for k in cat_df.keys():
#   cat_df = cat_df.drop(index=cat_df[cat_df[k].isna() == True].index)

# all_df = pd.merge(cat_df, qu_df, how="outer")

qu_df = qu_df.drop(columns=["participant_id"])



# cat_df = cat_df.drop(columns=["participant_id"])
# all_df = all_df.drop(columns=["participant_id"])

# for k in all_df.keys():
#   all_df = all_df.drop(index=all_df[all_df[k].isna() == True].index)


qu_df = (qu_df - qu_df.min())/(qu_df.max()-qu_df.min())
# cat_df = (cat_df - cat_df.min())/(cat_df.max()-cat_df.min())
# all_df = (all_df - all_df.min())/(all_df.max()-all_df.min())


participant_id                 object
ADHD_Outcome                    int64
Sex_F                           int64
EHQ_EHQ_Total                 float64
ColorVision_CV_Score            int64
APQ_P_APQ_P_CP                  int64
APQ_P_APQ_P_ID                  int64
APQ_P_APQ_P_INV                 int64
APQ_P_APQ_P_OPD                 int64
APQ_P_APQ_P_PM                  int64
APQ_P_APQ_P_PP                  int64
SDQ_SDQ_Conduct_Problems        int64
SDQ_SDQ_Difficulties_Total      int64
SDQ_SDQ_Emotional_Problems      int64
SDQ_SDQ_Externalizing           int64
SDQ_SDQ_Generating_Impact       int64
SDQ_SDQ_Hyperactivity           int64
SDQ_SDQ_Internalizing           int64
SDQ_SDQ_Peer_Problems           int64
SDQ_SDQ_Prosocial               int64
MRI_Track_Age_at_Scan         float64
dtype: object


## TTV Split

In [3]:
read_in_dataset = True

In [4]:
data = qu_df.copy(deep=True)
print(data.keys())
print(data.isna().value_counts())
print(data.shape)

# Classify sex only
quSexX = data.drop(columns=["Sex_F", "ADHD_Outcome"])
quSexY = data["Sex_F"]
XSex_train, XSex_test, ySex_train, ySex_test = train_test_split(quSexX, quSexY, test_size=0.2)



# Classify ADHD only
quADHDX = data.drop(columns=["Sex_F", "ADHD_Outcome"])
quADHDY = data["ADHD_Outcome"]
XADHD_train, XADHD_test, yADHD_train, yADHD_test = train_test_split(quADHDX, quADHDY, test_size=0.2)


Index(['ADHD_Outcome', 'Sex_F', 'EHQ_EHQ_Total', 'ColorVision_CV_Score',
       'APQ_P_APQ_P_CP', 'APQ_P_APQ_P_ID', 'APQ_P_APQ_P_INV',
       'APQ_P_APQ_P_OPD', 'APQ_P_APQ_P_PM', 'APQ_P_APQ_P_PP',
       'SDQ_SDQ_Conduct_Problems', 'SDQ_SDQ_Difficulties_Total',
       'SDQ_SDQ_Emotional_Problems', 'SDQ_SDQ_Externalizing',
       'SDQ_SDQ_Generating_Impact', 'SDQ_SDQ_Hyperactivity',
       'SDQ_SDQ_Internalizing', 'SDQ_SDQ_Peer_Problems', 'SDQ_SDQ_Prosocial',
       'MRI_Track_Age_at_Scan'],
      dtype='object')
ADHD_Outcome  Sex_F  EHQ_EHQ_Total  ColorVision_CV_Score  APQ_P_APQ_P_CP  APQ_P_APQ_P_ID  APQ_P_APQ_P_INV  APQ_P_APQ_P_OPD  APQ_P_APQ_P_PM  APQ_P_APQ_P_PP  SDQ_SDQ_Conduct_Problems  SDQ_SDQ_Difficulties_Total  SDQ_SDQ_Emotional_Problems  SDQ_SDQ_Externalizing  SDQ_SDQ_Generating_Impact  SDQ_SDQ_Hyperactivity  SDQ_SDQ_Internalizing  SDQ_SDQ_Peer_Problems  SDQ_SDQ_Prosocial  MRI_Track_Age_at_Scan
False         False  False          False                 False           False     

## Random Forest

In [5]:
n_estimators = 800
max_depth = 2
rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)

param_dist = {'n_estimators': randint(50,500),
              'max_depth': randint(1,20)}
rand_search = RandomizedSearchCV(rf, param_distributions=param_dist, n_iter=8,cv=5)


In [6]:
# Sex Classification
rand_search.fit(XSex_train, ySex_train)
y_pred = rand_search.predict(XSex_test)
accuracy = accuracy_score(ySex_test, y_pred)
precision = precision_score(ySex_test, y_pred)
recall = recall_score(ySex_test, y_pred)
print("Sex-Based Accuracy:", accuracy)
print("Sex-Based Precision:", precision)
print("Sex-Based Recall:", recall)

Sex-Based Accuracy: 0.656441717791411
Sex-Based Precision: 0.6363636363636364
Sex-Based Recall: 0.22580645161290322


In [7]:
# ADHD Classification
rand_search.fit(XADHD_train, yADHD_train)
y_pred = rand_search.predict(XADHD_test)
accuracy = accuracy_score(yADHD_test, y_pred)
precision = precision_score(yADHD_test, y_pred)
recall = recall_score(yADHD_test, y_pred)
print("ADHD-Based Accuracy:", accuracy)
print("ADHD-Based Precision:", precision)
print("ADHD-Based Recall:", recall)

ADHD-Based Accuracy: 0.8404907975460123
ADHD-Based Precision: 0.8384615384615385
ADHD-Based Recall: 0.956140350877193


Random Forest Classifier does best when classifying ADHD only

## Logistic Regression

In [8]:
from sklearn.linear_model import LogisticRegression

logReg = LogisticRegression(random_state=73)

In [9]:
# Sex Classification
logReg.fit(XSex_train, ySex_train)
y_pred = logReg.predict(XSex_test)
accuracy = accuracy_score(ySex_test[ySex_test==1], y_pred[ySex_test==1])
precision = precision_score(ySex_test, y_pred)
recall = recall_score(ySex_test, y_pred)
print("Sex-Based Accuracy:", accuracy)
print("Sex-Based Precision:", precision)
print("Sex-Based Recall:", recall)
print(y_pred[ySex_test==1])
# print(ySex_test[])

Sex-Based Accuracy: 0.1774193548387097
Sex-Based Precision: 0.6111111111111112
Sex-Based Recall: 0.1774193548387097
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0.
 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 1. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0.]


In [10]:
# ADHD Classification
logReg.fit(XADHD_train, yADHD_train)
y_pred = logReg.predict(XADHD_test)
accuracy = accuracy_score(yADHD_test[yADHD_test==1], y_pred[yADHD_test==1])
precision = precision_score(yADHD_test, y_pred)
recall = recall_score(yADHD_test, y_pred)
print("ADHD-Based Accuracy:", accuracy)
print("ADHD-Based Precision:", precision)
print("ADHD-Based Recall:", recall)
print(y_pred)
print(yADHD_test.values)

ADHD-Based Accuracy: 0.9385964912280702
ADHD-Based Precision: 0.8492063492063492
ADHD-Based Recall: 0.9385964912280702
[1. 1. 0. 1. 1. 1. 1. 0. 0. 1. 0. 1. 0. 0. 1. 1. 0. 1. 1. 0. 1. 1. 1. 1.
 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 0. 1. 0. 0. 1. 0. 1. 1. 1.
 1. 0. 1. 0. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 0. 1. 0.
 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 0. 1.
 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 0. 0. 1. 1. 0. 1.
 1. 1. 0. 0. 1. 0. 0. 1. 1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[1. 1. 0. 1. 0. 1. 0. 0. 0. 1. 1. 1. 0. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 0.
 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 0. 1. 0. 0. 1. 0. 1. 1. 1.
 0. 1. 1. 1. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0. 1. 1. 1. 0. 1. 0.
 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 0. 0.
 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 0. 1. 1. 1. 0. 0. 1. 1. 0. 0. 1. 0. 0. 1.
 1. 1. 0. 0. 1. 1. 

In [ ]:
prediction_sex = pd.read_excel("OUTPUTS.xlsx")
sub_test_data = pd.read_excel("dataset/widsdatathon2025/TEST/TEST_QUANTITATIVE_METADATA.xlsx")

sub_test_data = sub_test_data.drop(columns=["participant_id"])

for k in sub_test_data.keys():
  m = sub_test_data[k].median()
  sub_test_data[k] = sub_test_data[k].fillna(m)

sub_test_data = (sub_test_data - sub_test_data.min())/(sub_test_data.max()-sub_test_data.min())

sub_test_prediction = logReg.predict(sub_test_data)
print(sub_test_prediction)

prediction_sex.insert(1, "ADHD_Outcome", sub_test_prediction)

# prediction_sex.to_csv("FULL_PREDICTION.csv", index=False)

[1. 1. 0. 1. 1. 1. 0. 0. 1. 1. 0. 0. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 0. 1.
 1. 1. 0. 0. 0. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 0. 1. 0. 1. 1. 0. 1. 1. 0. 1. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 0. 1. 0. 0. 1. 1. 1.
 1. 0. 1. 0. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 0. 1. 0. 0. 1. 1. 1. 1.
 1. 1. 1. 1. 0. 0. 0. 1. 0. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 0. 1. 0. 0. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 0.
 1. 1. 1. 1. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 1. 1. 0. 0. 0.
 1. 0. 1. 0. 0. 1. 0. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1.
 0. 1. 1. 0. 1. 0. 1. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0.
 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 0. 1. 1. 0. 1. 1.]


Logistic regression appears the best out of the models tested

## SVM

In [ ]:
from sklearn import svm

svmm = svm.SVC(kernel='poly') # Linear Kernel

In [ ]:
# Sex Classification
svmm.fit(XSex_train, ySex_train)
y_pred = svmm.predict(XSex_test)
accuracy = accuracy_score(ySex_test, y_pred)
precision = precision_score(ySex_test, y_pred)
recall = recall_score(ySex_test, y_pred)
print("Sex-Based Accuracy:", accuracy)
print("Sex-Based Precision:", precision)
print("Sex-Based Recall:", recall)

Sex-Based Accuracy: 0.6809815950920245
Sex-Based Precision: 0.5
Sex-Based Recall: 0.25


In [ ]:
# ADHD Classification
svmm.fit(XADHD_train, yADHD_train)
y_pred = svmm.predict(XADHD_test)
accuracy = accuracy_score(yADHD_test, y_pred)
precision = precision_score(yADHD_test, y_pred)
recall = recall_score(yADHD_test, y_pred)
print("ADHD-Based Accuracy:", accuracy)
print("ADHD-Based Precision:", precision)
print("ADHD-Based Recall:", recall)

ADHD-Based Accuracy: 0.7975460122699386
ADHD-Based Precision: 0.8305084745762712
ADHD-Based Recall: 0.8828828828828829
